# Finalize HotPotQA_Vi_1k


In [1]:
from pathlib import Path
import os
import json
import pandas as pd

# SỬA thành URL GitHub repo của bạn
REPO_URL = "https://github.com/chichic21039/Prepare-data-HotpotQA-VN.git"

REPO_ROOT = Path("/content/Prepare-data-HotpotQA-VN")

# Clone repo nếu runtime mới chưa có
if not REPO_ROOT.exists():
    !git clone {REPO_URL} {REPO_ROOT}
else:
    print("Repo already exists:", REPO_ROOT)

DATA_DIR = REPO_ROOT / "data" / "hotpotqa_vi_1k"

CHECKPOINT_DIR = DATA_DIR / "checkpoints"
REVIEW_DIR = DATA_DIR / "reviews"
FINAL_DIR = DATA_DIR / "final"

# final có thể chưa tồn tại trên GitHub vì Git không lưu folder rỗng
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Repo       :", REPO_ROOT)
print("Checkpoint :", CHECKPOINT_DIR)
print("Reviews    :", REVIEW_DIR)
print("Final      :", FINAL_DIR)

Cloning into '/content/Prepare-data-HotpotQA-VN'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 30 (delta 6), reused 22 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 5.08 MiB | 13.59 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Repo       : /content/Prepare-data-HotpotQA-VN
Checkpoint : /content/Prepare-data-HotpotQA-VN/data/hotpotqa_vi_1k/checkpoints
Reviews    : /content/Prepare-data-HotpotQA-VN/data/hotpotqa_vi_1k/reviews
Final      : /content/Prepare-data-HotpotQA-VN/data/hotpotqa_vi_1k/final


In [2]:
print("Checkpoint files:")
for x in CHECKPOINT_DIR.iterdir():
    print(" -", x.name)

print("\nReview files:")
for x in REVIEW_DIR.iterdir():
    print(" -", x.name)

Checkpoint files:
 - final_queries.json
 - selected_qrels.json
 - final_corpus.json

Review files:
 - priority_manual_review_completed.csv
 - type_a_manual_review_completed.csv


In [3]:
with open(
    CHECKPOINT_DIR / "final_queries.json",
    encoding="utf-8"
) as f:
    final_queries = json.load(f)

with open(
    CHECKPOINT_DIR / "final_corpus.json",
    encoding="utf-8"
) as f:
    final_corpus = json.load(f)

with open(
    CHECKPOINT_DIR / "selected_qrels.json",
    encoding="utf-8"
) as f:
    selected_qrels = json.load(f)

print("Queries:", len(final_queries))
print("Corpus :", len(final_corpus))
print("Qrels  :", len(selected_qrels))

Queries: 1000
Corpus : 9822
Qrels  : 2000


In [7]:
import pandas as pd
from charset_normalizer import from_path

priority_path = REVIEW_DIR / "priority_manual_review_completed.csv"
type_a_path = REVIEW_DIR / "type_a_manual_review_completed.csv"


def detect_encoding(path):
    result = from_path(path).best()
    return result.encoding if result is not None else "utf-8"


priority_encoding = detect_encoding(priority_path)
type_a_encoding = detect_encoding(type_a_path)

print("Priority encoding:", priority_encoding)
print("Type A encoding :", type_a_encoding)


priority_df = pd.read_csv(
    priority_path,
    encoding=priority_encoding
)

type_a_df = pd.read_csv(
    type_a_path,
    encoding=type_a_encoding
)

print("Priority:", len(priority_df))
print("Type A  :", len(type_a_df))


reviews = pd.concat(
    [priority_df, type_a_df],
    ignore_index=True
)

reviews["id"] = reviews["id"].astype(str)

print("Total reviewed:", len(reviews))
print("Unique IDs    :", reviews["id"].nunique())

Priority encoding: cp1250
Type A encoding : utf_8
Priority: 63
Type A  : 153
Total reviewed: 216
Unique IDs    : 216


In [8]:
duplicates = reviews[
    reviews["id"].duplicated(keep=False)
]

print("Duplicate review IDs:", len(duplicates))

if len(duplicates):
    display(duplicates)

Duplicate review IDs: 0


In [9]:
reviews_by_id = {
    str(row["id"]): row
    for _, row in reviews.iterrows()
}

applied = 0
kept = 0

for q in final_queries:

    qid = str(q["id"])

    if qid not in reviews_by_id:
        continue

    r = reviews_by_id[qid]

    corrected = r.get("answer_vi_corrected")
    notes = r.get("notes")

    corrected = (
        ""
        if pd.isna(corrected)
        else str(corrected).strip()
    )

    notes = (
        ""
        if pd.isna(notes)
        else str(notes).strip()
    )

    # lưu audit trail
    q["answer_vi_before_review"] = q["answer_vi"]
    q["manual_review_notes"] = notes

    if corrected:
        q["answer_vi"] = corrected
        q["answer_vi_method"] = "manual_review_corrected"
        applied += 1

    else:
        q["answer_vi_method"] = "manual_review_verified"
        kept += 1

    q["answer_vi_confidence"] = "high"
    q["answer_vi_needs_review"] = False


print("Corrected:", applied)
print("Verified unchanged:", kept)
print("Total processed:", applied + kept)

Corrected: 36
Verified unchanged: 180
Total processed: 216


In [10]:
source_issue_mask = reviews["notes"].fillna("").str.contains(
    "source_qa_issue",
    case=False
)

source_issues = reviews[source_issue_mask].copy()

print("Source QA issues:", len(source_issues))

source_issues.to_csv(
    FINAL_DIR / "source_qa_issues.csv",
    index=False,
    encoding="utf-8-sig"
)

Source QA issues: 5


In [11]:
question_issue_mask = reviews["notes"].fillna("").str.contains(
    "question_vi_issue",
    case=False
)

question_issues = reviews[
    question_issue_mask
].copy()

print("Question VI issues:", len(question_issues))

Question VI issues: 0


In [12]:
assert len(final_queries) == 1000

ids = [str(x["id"]) for x in final_queries]

assert len(ids) == len(set(ids)), "Duplicate query IDs!"

missing_answers = [
    x["id"]
    for x in final_queries
    if not str(x.get("answer_vi", "")).strip()
]

print("Missing answer_vi:", len(missing_answers))

assert len(missing_answers) == 0

Missing answer_vi: 0


In [13]:
qrel_query_ids = {
    str(x["query_id"])
    for x in selected_qrels
}

query_ids = {
    str(x["id"])
    for x in final_queries
}

print(
    "Qrel queries missing from queries:",
    len(qrel_query_ids - query_ids)
)

from collections import Counter

count_qrels = Counter(
    str(x["query_id"])
    for x in selected_qrels
)

print(
    "Qrels/query distribution:",
    Counter(count_qrels.values())
)

Qrel queries missing from queries: 0
Qrels/query distribution: Counter({2: 1000})


In [14]:
corpus_ids = set(
    str(x)
    for x in final_corpus.keys()
)

missing_gold = [
    x
    for x in selected_qrels
    if str(x["corpus_id"]) not in corpus_ids
]

print("Missing qrel corpus docs:", len(missing_gold))

assert len(missing_gold) == 0

Missing qrel corpus docs: 0


# Export dataset final

In [15]:
with open(
    FINAL_DIR / "corpus.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for doc in final_corpus.values():

        output = {
            "id": str(doc["id"]),
            "title": doc["title"],
            "text": doc["text"],
            "language": "vi"
        }

        f.write(
            json.dumps(
                output,
                ensure_ascii=False
            ) + "\n"
        )

In [16]:
with open(
    FINAL_DIR / "queries.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for q in final_queries:

        output = {
            "id": str(q["id"]),

            "question_vi": q["question_vi"],
            "question_en": q["question_en"],

            "answer_vi": q["answer_vi"],
            "answer_en": q["answer_en"],

            "type": q["type"],
            "level": q["level"],

            "supporting_facts":
                q["supporting_facts"]
        }

        f.write(
            json.dumps(
                output,
                ensure_ascii=False
            ) + "\n"
        )

In [17]:
qrels_df = pd.DataFrame(
    selected_qrels
)

qrels_df.to_csv(
    FINAL_DIR / "qrels.tsv",
    sep="\t",
    index=False
)

In [18]:
print("=" * 60)
print("HOTPOTQA-VI-1K FINAL")
print("=" * 60)

print("Queries :", len(final_queries))
print("Corpus  :", len(final_corpus))
print("Qrels   :", len(selected_qrels))

print(
    "Source QA issues:",
    len(source_issues)
)

print(
    "Question VI issues:",
    len(question_issues)
)

print("\nSaved to:")
print(FINAL_DIR)

HOTPOTQA-VI-1K FINAL
Queries : 1000
Corpus  : 9822
Qrels   : 2000
Source QA issues: 5
Question VI issues: 0

Saved to:
/content/Prepare-data-HotpotQA-VN/data/hotpotqa_vi_1k/final
